Part B – Question 1
Random Forest

dataset

Decision Tree

In [1]:
import numpy as np
from datasets import load_dataset

dataset = load_dataset("fashion_mnist")

X_train = np.array(dataset['train']['image']).reshape(-1,28*28) / 255
y_train = np.array(dataset['train']['label'])

X_test = np.array(dataset['test']['image']).reshape(-1,28*28) / 255
y_test = np.array(dataset['test']['label'])

print(X_train.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


(60000, 784)


Train Single Decision Tree

In [2]:
def softmax(z):

    exp = np.exp(z - np.max(z, axis=1, keepdims=True))

    return exp / np.sum(exp, axis=1, keepdims=True)

Random Forest

In [3]:
num_classes = 10

def one_hot(y):
    Y = np.zeros((len(y), num_classes))
    for i in range(len(y)):
        Y[i, y[i]] = 1
    return Y

Y_train = one_hot(y_train)

n_features = X_train.shape[1]

W = np.zeros((n_features, num_classes))
b = np.zeros((1, num_classes))

lr = 0.1
epochs = 50

for epoch in range(epochs):

    logits = X_train @ W + b

    probs = softmax(logits)

    grad_W = X_train.T @ (probs - Y_train) / len(X_train)

    grad_b = np.mean(probs - Y_train, axis=0)

    W -= lr * grad_W
    b -= lr * grad_b

    if epoch % 10 == 0:
        loss = -np.mean(Y_train * np.log(probs + 1e-9))
        print("epoch", epoch, "loss", loss)

epoch 0 loss 0.23025850829940459
epoch 10 loss 0.13028340935649704
epoch 20 loss 0.10566481170804656
epoch 30 loss 0.09435812214909296
epoch 40 loss 0.087579575791492


Train Random Forest

In [4]:
logits = X_test @ W + b

pred = np.argmax(logits, axis=1)

accuracy = np.mean(pred == y_test)

print("Test Accuracy:", accuracy)

Test Accuracy: 0.7273


Compare Performance

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import mean_squared_error

lin = LinearRegression()

lin.fit(X_train, y_train)

pred_lin = lin.predict(X_test)

mse_lin = mean_squared_error(y_test, pred_lin)

print("Linear Regression MSE:", mse_lin)

Linear Regression MSE: 1.9684297142457847


In [ ]:
kr_rbf = KernelRidge(kernel='rbf', gamma=0.1)

kr_rbf.fit(X_train, y_train)

pred_rbf = kr_rbf.predict(X_test)

mse_rbf = mean_squared_error(y_test, pred_rbf)

print("Kernel Ridge (RBF) MSE:", mse_rbf)

In [ ]:
import numpy as np
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import mean_squared_error

num_classes = 10

def one_hot(y):
    Y = np.zeros((len(y), num_classes))
    for i in range(len(y)):
        Y[i, y[i]] = 1
    return Y

Y_train = one_hot(y_train)

kr_poly = KernelRidge(kernel='polynomial', degree=3)

kr_poly.fit(X_train, y_train)

pred_poly = kr_poly.predict(X_test)

mse_poly = mean_squared_error(y_test, pred_poly)

print("Kernel Ridge (Polynomial) MSE:", mse_poly)